In [0]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 108.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = ""

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:27<00:00, 169MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-abe3f841-66fc-4467-a85c-a4 nogroup 8.4G Jan 16 10:48 2019-Nov.csv
-rwxrwxrwx 1 spark-abe3f841-66fc-4467-a85c-a4 nogroup 5.3G Jan 16 10:51 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 10:35 delta
-rwxrwxrwx 1 spark-abe3f841-66fc-4467-a85c-a4 nogroup 4.3G Jan 16 10:48 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 10:35 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-abe3f841-66fc-4467-a85c-a4 nogroup 8.4G Jan 16 10:48 2019-Nov.csv
-rwxrwxrwx 1 spark-abe3f841-66fc-4467-a85c-a4 nogroup 5.3G Jan 16 10:51 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 10:35 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 16 10:35 outputs


In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
csv_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events_oct2019"

db = "workspace.ecommerce"
table_managed = f"{db}.events_oct2019_managed"
table_external = f"{db}.events_oct2019_external"

events = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(csv_path))

events = (events
          .withColumn("price", F.col("price").cast("double")))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_gold")
base_vol = "/Volumes/workspace/ecommerce/ecommerce_data"

raw_csv = f"{base_vol}/2019-Oct.csv"

bronze_path = f"{base_vol}/delta/bronze/events"
silver_path = f"{base_vol}/delta/silver/events"
gold_path   = f"{base_vol}/delta/gold/product_perf"


In [0]:
bronze = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(raw_csv)
          .withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("source_file", F.lit(raw_csv)))

(bronze.write
 .format("delta")
 .mode("overwrite")
 .save(bronze_path))

print("Bronze written to:", bronze_path)

Bronze written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events


In [0]:
(bronze.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_bronze.events"))

In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

silver = (bronze_df
          .withColumn("event_ts", F.to_timestamp("event_time"))
          .withColumn("event_date", F.to_date("event_ts"))
          .withColumn("price", F.col("price").cast("double"))
          .filter(F.col("event_ts").isNotNull())
          .filter(F.col("event_type").isNotNull())
          .filter(F.col("user_session").isNotNull())
          .filter((F.col("price").isNull()) | ((F.col("price") > 0) & (F.col("price") < 10000)))
          .dropDuplicates(["user_session", "event_time", "event_type", "product_id"])
          .withColumn(
              "price_tier",
              F.when(F.col("price").isNull(), F.lit("unknown"))
               .when(F.col("price") < 10, F.lit("budget"))
               .when(F.col("price") < 50, F.lit("mid"))
               .otherwise(F.lit("premium"))
          )
         )

(silver.write
 .format("delta")
 .mode("overwrite")
 .save(silver_path))

print("Silver written to:", silver_path)

Silver written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events


In [0]:
(silver.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_silver.events"))

In [0]:
silver_df = spark.read.format("delta").load(silver_path)

product_perf = (
    silver_df.groupBy("product_id")
    .agg(
        F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("unique_viewers"),
        F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("unique_purchasers"),
        F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))), 2).alias("revenue")
    )
    .withColumn(
        "conversion_rate_pct",
        F.when(F.col("unique_viewers") == 0, F.lit(0.0))
         .otherwise(F.round((F.col("unique_purchasers") / F.col("unique_viewers")) * 100, 4))
    )
)

(product_perf.write
 .format("delta")
 .mode("overwrite")
 .save(gold_path))

print("Gold written to:", gold_path)

display(product_perf.orderBy(F.col("revenue").desc()).limit(20))

Gold written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf


product_id,unique_viewers,unique_purchasers,revenue,conversion_rate_pct
1005115,170989,8352,1.240483595E7,4.8845
1005105,114813,4794,1.023924868E7,4.1755
1004249,96989,5538,6729380.83,5.7099
1005135,62646,2163,5567806.64,3.4527
1004767,175572,14410,5430222.72,8.2075
1002544,89025,6781,4854785.55,7.617
1004856,197840,19228,3798168.71,9.719
1002524,51704,4132,3538299.12,7.9916
1003317,56575,2179,3051294.26,3.8515
1004870,84318,7331,3027098.05,8.6945


In [0]:
(product_perf.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_gold.product_perf"))